# Assignment 2 $-$ Argument Acquisition
_Solutions have to be submitted in pairs of two by Monday, May 11th, 23:59 (UTC+2)._

**Grading:**
- This assignment is worth 30 points, you get points for running code without unhandled, preventable errors, correct outputs and answers to analysis questions
- **0 - *Fail*:** No submission; non-working code (preventable errors and exceptions); code that directly contradicts the task description or produces incorrect outputs; authorship violations like plagiarism or solutions fully or largely generated by AI

**Submission Components:**

- **Code:** Tasks 1 and 2
  - **Output Files:** Your submission should include the output files in the correct folder with the correct naming.
  - **Explanation:** You should add overall explanations of your code (i.e., modify/employ docstrings) and comments for individual implementation decisions.
- **Analysis:** Task 3 answered in full text with sensible formatting and using the results from task 2 explicitly.


**Submission Group:**
Ole Rößler,
TODO

In [2]:
import bz2
import json 
import pandas as pd
import numpy as np

## Assignment Goals

### Motivation
To study arguments, we need arguments, which we can get from many places: comment sections (on news websites/social media), discussion or debate forums, deliberative democracy platforms, parliamentary proceedings (speeches, debates), talk shows ...

Acquiring and manually processing new data takes time and also often money, e.g., for manual annotation studies. Before collecting new data, it is thus usually worth checking what already exists and whether you are allowed to use it. Below is a simple workflow to finding data for any research project:

1. Check for existing datasets
    - Papers related to your topic (methods or goals)
    - Survey papers compiling datasets and resources on one topic
    - Platforms like Kaggle, HuggingfaceHub (datasets you find here often come with exact instructions and guides)
2. No luck? $\to$ Check usage restrictions
    - `robots.txt` files show bots which subpages (not) to crawl, below is an example from [kialo](https://kialo.com/robots.txt))
       ```
       User-agent: Googlebot
       Disallow:
       User-agent: *
       Disallow: /
       ```
    - API terms (e.g., [data API terms](https://redditinc.com/policies/data-api-terms))
3. If unsure, ask:
    - Check with the rights holders directly (e.g., contact form, email).
    - Never illicitly use data: you cannot publish your data/findings without academic/legal consequences, but most importantly, we do not want to exacerbate the problem of data hungry AI-companies using data that is private or copyrighted.

### Application

We now apply this to a common task in computational argumentation: **argument quality assessment**.

Process argumentative discussions scraped from [r/ChangeMyView](https://www.reddit.com/r/changemyview/) by [Tan et al. (2016)](https://doi.org/10.1145/2872427.2883081) into a dataset usable for the task. On the subreddit, people can post their genuine opinions (OPs) to be persuaded of the opposite. If they are persuaded, they award a $\Delta$ point to the persuasive comment.

In this assignment, you will create a new dataset suitable for training a model to perform the following task:

> **Persuasiveness Classification:** Given an argumentative text, predict its persuasiveness, i.e., if it was awarded a $\Delta$.
> 
> **Example:**
>
> Because of the fact that the Vice President is responsible for Presidential duties when the President is away means that they are the perfect person to take over for them. In the event of whatever emergency would surround the need for a VP to take the President's spot, the experience that the VP already has with their duties would mean they could easily step into the President's place without disrupting the rest of the cabinet, all of whom would likely be very busy with their current departments.
>
> $\Delta$
> 
> (OP Thread: *CMV: The position of Vice President of the United States should be eliminated from our government.*)

### Dataset Structure
The data was scraped from Reddit for [Tan et al. (2016)](https://doi.org/10.1145/2872427.2883081), thus, its format and encoded information directly match the output of the [Reddit API](https://redditinc.com/policies/data-api-terms). 

Start by exploring the data:

1. Visit the dataset's [homepage](https://chenhaot.com/papers/changemyview.html) to consult the Readme and Blog tutorial
2. Follow these instructions to extract the data from `cmv_reddit_api_data.jsonlist.bz2`, which is saved in `./data`.
3. Explore the data: what does an instance look like, which information is encoded, and how are comment threads stored?
   
Move on to the graded tasks once you are familiar with the dataset structure.

In [3]:
data_path = "data/cmv_reddit_api_data.jsonlist.bz2"

# Set DEBUG to False to remove debug comments
DEBUG = False

def PRINT_DEBUG(msg=""):
    if DEBUG:
        if len(msg) > 0:
            print(f"*** {msg} ***")
        else:
            print()


# IGNORE: This only gives a brief intuition on how a line of the file actually looks like
with bz2.open(data_path, "rt") as f:
    """
    Excerpt from the Dataset-READ.ME file:
    `
    all/
        train_period_data.jsonlist.bz2
        heldout_period_data.jsonlist.bz2
    
        This directory contains all of the discussion in /r/ChangeMyView during the monitored period, as extracted from the Reddit API. Both files have the same format and store data for the training period (2013/01/01-2015/05/07) and the heldout period (2015/05/08-2015/09/01) respectively. Each line is a json object that includes all information for a submission. Each submission has the following fields:
            domain, banned_by, media_embed, subreddit, selftext_html, selftext, likes, suggested_sort, user_reports, secure_media, link_flair_text, id, from_kind, gilded, archived, clicked, report_reasons, author, media, comments, name, score, approved_by, over_18, hidden, thumbnail, subreddit_id, edited, link_flair_css_class, author_flair_css_class, downs, mod_reports, secure_media_embed, saved, removal_reason, stickied, from, is_self, from_id, permalink, hide_score, created, url, author_flair_text, quarantine, title, created_utc, ups, num_comments, visited, num_reports, distinguished.
        All fields except "comments" come directly from Reddit API. "comments" is a list of replies to the submission. Each reply has the following fields:
            subreddit_id, banned_by, removal_reason, link_id, likes, replies, user_reports, saved, id, gilded, archived, report_reasons, author, parent_id, score, approved_by, controversiality, body, edited, author_flair_css_class, downs, body_html, subreddit, score_hidden, name, created, author_flair_text, created_utc, ups, mod_reports, num_reports, distinguished.
    `
    """
    d = f.readline()
    PRINT_DEBUG(f"Extracted first line is {type(d)}, of length {len(d)}")
    PRINT_DEBUG()
    PRINT_DEBUG(f"Extraced json from line if {type(json.loads(d))}, of length {len(json.loads(d))}")
    PRINT_DEBUG()
    PRINT_DEBUG(f"Extraced keys from frist line are: {json.loads(d).keys()}")
    PRINT_DEBUG()
    PRINT_DEBUG(f"First post is:\n {json.dumps(json.loads(d), indent=2)}")    
    PRINT_DEBUG()
    PRINT_DEBUG(f"The post has {len(json.loads(d)['comments'])} additional comments with each {len(json.loads(d)['comments'][0])} keys: {json.loads(d)['comments'][0].keys()}")
    PRINT_DEBUG()
    PRINT_DEBUG(f"First two comments from the comment section:\n {json.dumps(json.loads(d)['comments'], indent=2)}")    
    PRINT_DEBUG()
    if False:
        PRINT_DEBUG(f"Save first post to file for better readability")
        with open("data/extracted_first_post.json", "w") as f_out:
            json.dump(json.loads(d), f_out, indent=2)
        PRINT_DEBUG(f"Saved first post to data/extracted_first_post.json")

In [4]:
def load_jsonl_bz2(filepath:str) -> list[dict]:
    """
    Load the entire jsonlist.bz2 file into a list of JSON dicts.
    
    Args:
        filepath (str): path to the jsonlist.bz2 file that should be read
    
    Returns:
        list[dict]: list of extracted posts
    """
    with bz2.open(filepath, "rt") as f:
        return [json.loads(line) for line in f]

PRINT_DEBUG("Loading Data")
data = load_jsonl_bz2(data_path)
PRINT_DEBUG(f"Loaded a list of {len(data)} posts containing discussion in /r/ChangeMyView")

---

## Task 1

Extract all the posts from all the original threads into one dataset and save it.
- You should only include the following information: _post\_id, parent\_id, thread\_id, author, text, impact_votes, impact_persuasion_, and _persuasiveness_.
    - The first five features are identically named in the data, but the _impact_ features use the upvotes and author flair, respectively, and should be saved as positive integers (_- 1 point each_).
    - The _persuasiveness_ must be extracted as a $\Delta$ in the comment's replies and saved as a binary score (0 or 1) (_- 3 points_).
- Add all comments to a `pandas.DataFrame` with column names _exactly_ matching the above list of features (_- 1 point per error_). 
- Make sure that you only include posts that are usable for prediction, i.e., posts with unique, non-empty text fields (_- 3 points_).
- Save the DataFrame to a cvs `./outputs/posts.csv` file _without_ including the index (_1 point_).

<div style="text-align: right"><b>10 points</b></div>

In [ ]:
required_comment_keys = {
    "body",
    "id",
    "link_id",
    "parent_id",
    "author",
    "ups",
    "author_flair_text"
}

# TODO refactor to use the dataframe form the beginning
all = []
for d in data:
    post = {}
    post["text"] = d["selftext"]
    post["post_id"] = d["name"]
    post["thread_id"] = d["name"]
    post["parent_id"] = None
    post["author"] = d["author"]
    post["impact_votes"] = d["ups"]
    post["impact_persuasion"] = d["author_flair_text"]
    post["persuasiveness"] = 0
    all.append(post)

    # Map IDs only for the current thread so we don't update posts/comments from prior threads
    thread_map = {post["post_id"]: len(all) - 1}
    
    for c in d["comments"]:
        if not required_comment_keys.issubset(c):
            continue

        # All DeltaBot comments represent a rewarded delta
        if c["author"] == "DeltaBot":
            if c["parent_id"] in thread_map:
                post_id = all[thread_map[c["parent_id"]]]["parent_id"]
                all[thread_map[post_id]]["persuasiveness"] = 1
                continue
            raise ValueError(f"Warning: DeltaBot comment with id {c['id']} has parent_id {c['parent_id']} that is not in the thread map.")

        comment = {}
        comment["text"] = c["body"] 
        comment["post_id"] = c["name"]
        comment["thread_id"] = c["link_id"]
        comment["parent_id"] = c["parent_id"]
        comment["author"] = c["author"]
        comment["impact_votes"] = c["ups"]
        comment["impact_persuasion"] = c["author_flair_text"]
        comment["persuasiveness"] = 0

        all.append(comment)
        thread_map[comment["post_id"]] = len(all) - 1


df = pd.DataFrame(all)

# remove all duplicate comments
df.drop_duplicates(subset=["text"], keep=False, inplace=True)

# replace deleted authors with None
df.loc[df["author"] == "[deleted]", "author"] = None

# remove dangling comments as they cannot be properly contextualized
df = df[df["parent_id"].isin(df["post_id"]) | df["parent_id"].isna()]

,text,post_id,thread_id,parent_id,author,impact_votes,impact_persuasion,persuasiveness
50,"Sorry no-mad, your comment has been removed: \...",t1_cunfwmv,t3_3j8yfq,t1_cuncjbv,Grunt08,1,89∆,0
70,"Aw, thanks! And I just edited my post after yo...",t1_cune5ow,t3_3j8yfq,t1_cunduof,ghoooooooooost,1,1∆,0
71,You can give more than 1 delta out.,t1_cunf0rz,t3_3j8yfq,t1_cunduof,Zouavez,1,NaN,0
212,"Sorry sweetmercy, your comment has been remove...",t1_cuneqhc,t3_3j8bhs,t1_cundn8f,huadpe,1,132∆,0
217,"Sorry Zijndarling, your comment has been remov...",t1_cun3r6n,t3_3j8bhs,t1_cun3h04,hacksoncode,1,119∆,0
...,...,...,...,...,...,...,...,...
145478,I honestly don't see anything wrong with it. ...,t1_cr2vlo6,t3_35blwd,t1_cr2vfu6,call_it_art,2,NaN,0
145479,"Sorry cde458, your comment has been removed: \...",t1_cr33sw1,t3_35blwd,t1_cr2vfu6,huadpe,1,132∆,0
145525,"Sorry mojo_magnifico, your comment has been re...",t1_cr33ow9,t3_35blwd,t1_cr31ggg,huadpe,1,132∆,0
145597,&gt;There is absolutely nothing wrong with sta...,t1_cr2vnos,t3_35bc4b,t1_cr2uk7u,WumboWombo,1,NaN,0


In [12]:
df

,text,post_id,thread_id,parent_id,author,impact_votes,impact_persuasion,persuasiveness
0,"From what I understand, the only significant d...",t3_3j90qv,t3_3j90qv,NaN,TentacleBird,0,1∆,0
1,The vice president often is called to act as a...,t1_cun9x23,t3_3j90qv,t3_3j90qv,draculabakula,1,14∆,0
2,"Your first point is very strong, however, noth...",t1_cunbngs,t3_3j90qv,t1_cun9x23,TentacleBird,1,1∆,0
3,Because of the fact that the Vice President is...,t1_cuncccq,t3_3j90qv,t1_cunbngs,EagenVegham,1,1∆,1
4,∆\n\nI realize now that the VP's duties can't ...,t1_cuncmdn,t3_3j90qv,t1_cuncccq,TentacleBird,1,1∆,0
...,...,...,...,...,...,...,...,...
145595,my genuine constructive advice is to make a be...,t1_cr4nqry,t3_35bc4b,t1_cr2wwo2,NorbitGorbit,1,8∆,0
145597,&gt;There is absolutely nothing wrong with sta...,t1_cr2vnos,t3_35bc4b,t1_cr2uk7u,WumboWombo,1,NaN,0
145599,"&gt;No it's not.\n\nSure, let's contradict eac...",t1_cr2ycwv,t3_35bc4b,t1_cr2x9tj,WumboWombo,1,NaN,0
145600,&gt; I've seen studies[1] that show college ...,t1_cr33198,t3_35bc4b,t1_cr2ycwv,Kirkaine,1,11∆,0


In [6]:
df["text"] = df["text"].astype(str)
df["post_id"] = df["post_id"].astype(str)
df["thread_id"] = df["thread_id"].astype(str)
df["parent_id"] = df["parent_id"].astype(str)
df["author"] = df["author"].astype(str)

# replace negatives impact_votes with 0, and convert to int
df["impact_votes"] = df["impact_votes"].apply(lambda x: max(x, 0)).astype(int) 

# remove weird characters, replace impute missing values with 0, then convert to int
df["impact_persuasion"] = df["impact_persuasion"].apply(
    lambda x: str(x).replace("∆", "").replace("Δ", "").replace("âˆ†", "")
)
df["impact_persuasion"] = df["impact_persuasion"].replace("nan", 0).astype(int)

# convert persuasiveness to int
df["persuasiveness"] = df["persuasiveness"].astype(int)


df

,text,post_id,thread_id,parent_id,author,impact_votes,impact_persuasion,persuasiveness
0,"From what I understand, the only significant d...",t3_3j90qv,t3_3j90qv,NaN,TentacleBird,0,1,0
1,The vice president often is called to act as a...,t1_cun9x23,t3_3j90qv,t3_3j90qv,draculabakula,1,14,0
2,"Your first point is very strong, however, noth...",t1_cunbngs,t3_3j90qv,t1_cun9x23,TentacleBird,1,1,0
3,Because of the fact that the Vice President is...,t1_cuncccq,t3_3j90qv,t1_cunbngs,EagenVegham,1,1,1
4,∆\n\nI realize now that the VP's duties can't ...,t1_cuncmdn,t3_3j90qv,t1_cuncccq,TentacleBird,1,1,0
...,...,...,...,...,...,...,...,...
145593,If you are so certain the economy will fail ca...,t1_cr2uooe,t3_35bc4b,t3_35bc4b,NorbitGorbit,0,8,0
145594,I was asking a question seeking advice for a v...,t1_cr2wwo2,t3_35bc4b,t1_cr2uooe,WumboWombo,2,0,0
145595,my genuine constructive advice is to make a be...,t1_cr4nqry,t3_35bc4b,t1_cr2wwo2,NorbitGorbit,1,8,0
145600,&gt; I've seen studies[1] that show college ...,t1_cr33198,t3_35bc4b,t1_cr2ycwv,Kirkaine,1,11,0


---

## Task 2 $-$ Data Exploration

Now that we have the data, we can analyze discussion and interaction patterns with regards to persuasiveness. For this, we first need to calculate relevant metrics and statistics. In the following task, you will calculate different data statistics from your extracted DataFrame and then display them in a sensible way.

### 2.1 Persuasion Patterns
First we want to find patterns in persuasion behavior. For this we will divide the threads into successful from unsuccessful persuasion attempts based on the occurrence of $\Delta$'s in the threads. Then, we can compare user behavior between these two sets.

1. What is the ratio of threads with at least one $\Delta$-awarded comment? (_1 point_)
2. What is the average number of the original poster's (OP) responses and its standard deviation? (_1 point_)
3. What is the average number of distinct users in a thread and its standard deviation? (_1 point_)

As the exact number of these metrics may be important to find patterns, a table visualization is superior to a plot in this case. Please add the thread ratio and averages (not the standard deviation) to a table with columns _persuasive_ and _non-persuasive_, and with row indices _ratio_, _OP responses_, and _distinct users_. Use the `DataFrame.to_markdown()` method to build the table and round floats to three decimal points (_2 points_). No formatting/rounding should be done to the data beforehand.

### 2.2 Persuasion Effort
For the next two metrics, use only those threads with at least one $\Delta$, i.e., those in the _persuasive_ set above. Calculate how long it takes until a $\Delta$ point is awarded in a given thread. Calculate this persuasion effort in two ways: 

1. What is the distribution and average depth of the $\Delta$-awarded comments? The depth is the number of parent comments to reach from the answer to the OP.
2. How often does the $\Delta$-awarded _user_ post (overall and on average) in the corresponding thread? If multiple users in the same thread were awarded $\Delta$ points, treat each _\<user>-\<thread>_ pair as its own instance to count answers. 
3. What are the average impact ratings of a $\Delta$-awarded comment/user and its standard deviation? If multiple users in the same thread were awarded $\Delta$ points, treat each _\<user>-\<thread>_ pair as its own instance to count answers. 

Calculate all statistics (_3 points_) and plot the first two results in aligned subfigures (_2 points_). Each figure should show a histogram of the $\Delta$ depth/user comment count and further include the average depth/count as an additional line in the plot. The plots should have titles, as well as axis descriptions. For question 3, calculate the statistics and keep them for the next task. 


<div style="text-align: right"><b>10 points</b></div>

In [7]:
# YOUR CODE HERE
raise NotImplementedError

NotImplementedError: 

---

## Task 3 $-$ Data Analysis
As we now have metrics to base our analysis and interpretation of data patterns on, we can explore the dataset fully. For both subtasks of task 2, analyze and interpret your observations (_5 points each_). For this, you should adhere to the argument structure below:
1. **Claim** $-$ Verbalize and summarize the overall findings from the subtask (*2 points*).
2. **Evidence** $-$ Include specific numbers from task 2. Copy the markdown table from 2.1 into your answer here (*1 point*).
3. **Interpretation** $-$ Reason about why this observation might be happening and what it may mean about persuasive behavior (*2 points*).

<div style="text-align: right"><b>10 points</b></div>

YOUR ANSWER HERE